# 01 — Explore catalogs
**Purpose:** work out which FTW files we need before downloading anything.
- **FTW-v2 benchmark-data**: one collection per country and year. It holds field polygons with crop labels, chips with spatially blocked train/val/test splits, and precomputed per-field AlphaEarth (AEF) and OlmoEarth embedding stats (mean and std of 64 dims over each field's 10 m pixels).
- **harmonized-field-data**: the source field-boundary declarations that the benchmark is cut from, in the fiboa schema. Checked only to confirm the benchmark already carries the same labels.
- **Not used:** `global-data-2e`, which has model-predicted boundaries with no crop labels and no embeddings.

**Output:** the tree files, the schemas, and the choice of one country/year with labels and embeddings for the same fields.

In [6]:
import json,urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin

In [ ]:
RAWDIR   = Path('../data/raw').mkdir(parents=True,exist_ok=True)
ROOTSDIR = {'benchmark':'https://data.source.coop/ftw/benchmark-data/catalog.json',
            'harmonized':'https://data.source.coop/ftw/harmonized-field-data/catalog.json'}

In [1]:
def get(url):
    with urllib.request.urlopen(url,timeout=60) as r:
        return json.load(r)

def node(url,depth):
    d = get(url)
    kids = [urljoin(url, l["href"]) for l in d.get("links", []) if l["rel"] == "child"]
    assets = {k: urljoin(url, a["href"]) for k, a in d.get("assets", {}).items()}
    return {"url": url, "id": d["id"], "type": d["type"], "title": d.get("title"),
            "assets": assets, "depth": depth, "kids": kids}

trees = {}
for name, root in ROOTS.items():
    r = node(root, 0)
    with ThreadPoolExecutor(16) as ex:
        lvl1 = list(ex.map(lambda u: node(u, 1), r["kids"]))
        lvl2 = list(ex.map(lambda u: node(u, 2), [k for n in lvl1 for k in n["kids"]]))
    trees[name] = [r] + lvl1 + lvl2
    lines = []
    for n in sorted(trees[name], key=lambda n: (n["url"].rsplit("/", 1)[0], n["depth"])):
        ind = "  " * n["depth"]
        lines.append(f"{ind}{n['id']} [{n['type']}] {n['title']}")
        lines += [f"{ind}    - {k}: {h}" for k, h in n["assets"].items()]
    (RAW / f"{name}_tree.txt").write_text("\n".join(lines))
    print(name, {d: sum(n["depth"] == d for n in trees[name]) for d in (0, 1, 2)})

benchmark {0: 1, 1: 37, 2: 180}
harmonized {0: 1, 1: 79, 2: 0}


In [2]:
# Summarize: embedding coverage in benchmark, crop-bearing collections in harmonized
import pandas as pd
bench = pd.DataFrame([n for n in trees["benchmark"] if n["depth"] == 2])
bench["has_emb"] = bench.assets.map(lambda a: "field_embeddings" in a)
bench["source"] = bench.url.str.split("/").str[-3]
print("benchmark collections:", len(bench), "| with field_embeddings:", bench.has_emb.sum())
print(bench[bench.has_emb].groupby("source").id.apply(list).to_string())
print("without embeddings:", sorted(bench.loc[~bench.has_emb, "id"]))

harm = pd.DataFrame([n for n in trees["harmonized"] if n["depth"] == 1])
harm["crops"] = harm.assets.map(lambda a: any("crop" in k for k in a))
print("\nharmonized collections:", len(harm), "| with crop style asset:", harm.crops.sum())
print(sorted(harm.loc[harm.crops, "id"]))

benchmark collections: 180 | with field_embeddings: 180
source
ai4sf_ml                                          [ai4sf_ml_2021]
at              [at_2018, at_2019, at_2020, at_2021, at_2022, ...
be_vlg          [be_vlg_2018, be_vlg_2019, be_vlg_2020, be_vlg...
be_wal                                              [be_wal_2022]
bg                  [bg_2021, bg_2022, bg_2023, bg_2024, bg_2025]
br_ba_lem                                        [br_ba_lem_2020]
ch              [ch_ag_2025, ch_ai_2025, ch_ar_2025, ch_be_202...
cz              [cz_2019, cz_2020, cz_2021, cz_2022, cz_2023, ...
de_nds                                              [de_nds_2025]
de_sh                        [de_sh_2023, de_sh_2024, de_sh_2025]
dk              [dk_2017, dk_2018, dk_2019, dk_2020, dk_2021, ...
ec_lt                                                [ec_lt_2021]
ee              [ee_2017, ee_2018, ee_2019, ee_2020, ee_2021, ...
es                                          [es_2025, es_po_2025]
es_cat       

In [3]:
# Print schemas + row counts of the candidate assets (metadata only, no data load)
import fsspec, pyarrow.parquet as pq
B = "https://data.source.coop/ftw/benchmark-data/ee/ee_2024/"
H = "https://data.source.coop/ftw/harmonized-field-data/ee/latest/ee.parquet"
assets = {"fields": B + "ee_2024_fields.parquet", "embeddings": B + "ee_2024_field_embeddings.parquet",
          "chips": B + "ee_2024_chips.parquet", "harmonized": H}
fs = fsspec.filesystem("https")
for k, u in assets.items():
    with fs.open(u, block_size=2**20) as f:
        pf = pq.ParquetFile(f)
        print(f"== {k}: rows={pf.metadata.num_rows:,} size={fs.size(u)/1e6:.0f} MB row_groups={pf.num_row_groups}")
        print(pf.schema_arrow.to_string(show_schema_metadata=False), "\n")

== fields: rows=166,500 size=278 MB row_groups=2
id: string
land_use: string
crop:name: string
hcat:name_en: string
crop:code: string
hcat:code: uint32
hcat:name: string
metrics:area: float
parcel_id: int64
geometry: binary
bbox: struct<xmin: double, ymin: double, xmax: double, ymax: double>
  child 0, xmin: double
  child 1, ymin: double
  child 2, xmax: double
  child 3, ymax: double 

== embeddings: rows=166,500 size=154 MB row_groups=2
id: string
pixels: int32
pixel_rule: string
aef_pixels: int32
aef_mean: list<element: float>
  child 0, element: float
aef_std: list<element: float>
  child 0, element: float
olmoearth_pixels: int32
olmoearth_mean: list<element: float>
  child 0, element: float
olmoearth_std: list<element: float>
  child 0, element: float 

== chips: rows=7,562 size=1 MB row_groups=1
gzd: string
mgrs_10km: string
id: string
field_coverage_pct: double
hcat_dominant_code: int64
hcat_dominant_name_en: string
hcat_dominant_pct: double
hcat_top: list<element: struct<code:

In [5]:
# Check id overlap (fields vs embeddings vs harmonized) and year metadata of embeddings/harmonized
def read_cols(u, cols):
    with fs.open(u, block_size=2**22) as f:
        return pq.read_table(f, columns=cols).to_pandas()
ids_f = read_cols(assets["fields"], ["id", "parcel_id"])
ids_e = read_cols(assets["embeddings"], ["id", "pixel_rule"])
ids_h = read_cols(assets["harmonized"], ["id", "parcel_id"])
print("unique ids f/e/h:", ids_f.id.nunique(), ids_e.id.nunique(), ids_h.id.nunique())
print("f∩e:", len(set(ids_f.id) & set(ids_e.id)), "| f∩h (id):", len(set(ids_f.id) & set(ids_h.id)),
      "| f∩h (parcel_id):", len(set(ids_f.parcel_id) & set(ids_h.parcel_id)))
print(ids_f.id.head(3).tolist(), ids_e.pixel_rule.value_counts().to_dict())
col = get(B + "collection.json"); hcol = get("https://data.source.coop/ftw/harmonized-field-data/ee/collection.json")
print("bench temporal:", col["extent"]["temporal"], "| version:", col.get("version"))
print("emb asset:", {k: v for k, v in col["assets"]["field_embeddings"].items() if k != "href"})
print("harm temporal:", hcol["extent"]["temporal"])

unique ids f/e/h: 166500 166500 166500
f∩e: 166500 | f∩h (id): 166500 | f∩h (parcel_id): 166500
['165356', '165355', '130916'] {'centre': 166266, 'touched': 234}
bench temporal: {'interval': [['2024-01-01T00:00:00Z', '2024-12-31T23:59:59Z']]} | version: 2.0.0-alpha.1
emb asset: {'type': 'application/vnd.apache.parquet', 'title': 'Per-field embedding statistics (AlphaEarth Foundations, OlmoEarth v1.3)', 'description': "Mean and standard deviation of each embedding dimension over each field's pixels, for 2024; joins onto the fields by `id`. AlphaEarth Foundations Satellite Embedding dataset by Google and Google DeepMind, CC-BY-4.0 (source of the aef_* columns). OlmoEarth v1.3 embeddings by Ai2 (Allen Institute for AI), OlmoEarth Artifact License, including its Section 2 use restrictions (source of the olmoearth_* columns).", 'file:size': 154310497, 'table:columns': [{'name': 'id', 'type': 'varchar', 'description': 'Field id; joins onto the `fields` (or `fields_filtered`) table.'}, {'name